# Checkpoint 5D — AMPDS2_01: five-seed regression LIME

Copy and run **only** the code cell below in the **existing** Colab runtime that passed Checkpoints 4B–5C. The cell verifies the exact D5 AMPds2 8,192-row reference, reproduces the saved seed-42 regression LIME result, then varies the LIME perturbation seed over 42, 123, 456, 789, 2026. Outputs are written under a new `CHECKPOINT5D` folder only after all gates pass. No model retraining, no SHAP or synthetic injection.

In [ ]:
# CHECKPOINT 5D — AMPDS2_01 five-seed regression LIME stability
# Only perturbation seed varies; detector and 8192-row training reference are frozen.
# No model fitting, threshold changes, SHAP, or historical overwrites.

from pathlib import Path
from itertools import combinations
import hashlib
import importlib.metadata as im
import json
import joblib
import numpy as np
import pandas as pd
from lime.lime_tabular import LimeTabularExplainer

required = ['ROOT', 'D5', 'FEATURES', 'COLUMNS', 'selected',
            'STABILITY_PROTOCOL', 'protocol_hash']
missing = [name for name in required if name not in globals()]
assert not missing, f'STOP: Missing prior verified runtime variables: {missing}'

# 1. Frozen environment, protocol, original feature order.
for pkg, version in {'scikit-learn':'1.6.1', 'lime':'0.2.0.1',
                     'numpy':'2.4.6', 'pandas':'2.2.3'}.items():
    assert im.version(pkg) == version, f'STOP: {pkg} version changed'
seeds = [42, 123, 456, 789, 2026]
p = STABILITY_PROTOCOL
canonical = json.dumps(p, sort_keys=True, separators=(',', ':'))
assert hashlib.sha256(canonical.encode('utf-8')).hexdigest() == protocol_hash
assert protocol_hash == '849e9d7ef15cc3b135ac74854f491ae3c2113f39c3ef527ca789cde8e829334c'
assert p['seeds'] == seeds
assert p['feature_order'] == list(FEATURES)
assert COLUMNS == [f'{f}_z' for f in FEATURES] and len(FEATURES) == 23
lp = p['lime']
assert lp['mode'] == 'regression' and lp['vary'] == 'perturbation_random_state'
assert lp['reference_size'] == 8192
assert lp['reference_seeds']['AMPds2'] == 2027
assert lp['num_samples'] == 5000 and lp['num_features'] == 10
assert lp['discretizer'] == 'quartile'
assert lp['feature_selection'] == 'auto'
assert lp['sample_around_instance'] is False
assert abs(float(lp['kernel_width']) - 0.75*np.sqrt(23)) < 1e-12

# 2. Historical D5 model, selected window, and saved explanation evidence.
D5 = Path(D5)
manifest = json.loads((D5/'manifest.json').read_text(encoding='utf-8'))
entry = manifest['model_and_background']['AMPds2']
model_path = Path(entry['model_path'])
score_path = Path(entry['scoring_file'])
assert model_path.is_file() and score_path.is_file()

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

assert file_sha256(model_path) == entry['model_sha256']
model = joblib.load(model_path)  # Previously verified user-owned reconstruction artifact.
assert model.n_features_in_ == 23 and len(model.estimators_) == 300
assert model.max_samples_ == 256 and abs(float(model.offset_) + .5) < 1e-12

def score_fn_amp(Z):
    Z = np.asarray(Z, dtype=np.float32)
    if Z.ndim == 1:
        Z = Z.reshape(1,-1)
    assert Z.ndim == 2 and Z.shape[1] == 23 and np.isfinite(Z).all()
    return -model.decision_function(Z)

case = selected.loc[selected['window_id'].eq('AMPDS2_01')]
assert len(case) == 1 and case.iloc[0]['dataset'] == 'AMPds2'
x_amp_lime = case[COLUMNS].to_numpy(dtype=np.float32)
assert x_amp_lime.shape == (1,23) and np.isfinite(x_amp_lime).all()
score = float(score_fn_amp(x_amp_lime)[0])
assert abs(score - float(case.iloc[0]['anomaly_score'])) <= 1e-9

fidelity_file = D5/'lime_fidelity_13_windows.csv'
coef_file = D5/'regression_lime_coefficients_130_rows.csv'
for path in [fidelity_file, coef_file]:
    assert path.is_file() and file_sha256(path) == manifest['artifact_sha256'][path.name], (
        f'STOP: Historical D5 evidence hash mismatch: {path.name}')
old_fidelity = pd.read_csv(fidelity_file)
old_fidelity = old_fidelity.loc[old_fidelity['window_id'].eq('AMPDS2_01')]
assert len(old_fidelity) == 1
old_fidelity = old_fidelity.iloc[0]
assert old_fidelity['lime_mode'] == 'regression'
assert abs(float(old_fidelity['lime_target_prediction'])-score) <= 1e-9
old_coef = pd.read_csv(coef_file)
old_coef = old_coef.loc[old_coef['window_id'].eq('AMPDS2_01')].sort_values('rank')
assert len(old_coef) == 10 and old_coef['rank'].tolist() == list(range(1,11))
assert old_coef['lime_mode'].eq('regression').all()
print('PASS: Protocol, frozen IF model, AMPDS2_01 target and historical D5 files')

# 3. Reconstruct the exact D5 8192-row reference from AMPds2 training data.
#    D5 Cell 6: seed 2027, sorted sampled positions, scoring-CSV training order.
N_TRAIN = 441483
REFERENCE_N = 8192
positions = np.sort(np.random.default_rng(2027).choice(
    N_TRAIN, size=REFERENCE_N, replace=False))
assert len(np.unique(positions)) == REFERENCE_N
reference_parts = []
training_seen = 0
for chunk in pd.read_csv(score_path, usecols=['split']+COLUMNS,
                         chunksize=75000, float_precision='round_trip',
                         low_memory=False):
    mask = chunk['split'].astype(str).eq('train')
    count = int(mask.sum())
    if count == 0:
        continue
    lower = np.searchsorted(positions, training_seen, side='left')
    upper = np.searchsorted(positions, training_seen+count, side='left')
    if upper > lower:
        local = positions[lower:upper]-training_seen
        reference_parts.append(
            chunk.loc[mask, COLUMNS].iloc[local].to_numpy(dtype=np.float32))
    training_seen += count
assert training_seen == N_TRAIN and len(reference_parts) > 0
reference_amp = np.vstack(reference_parts)
assert reference_amp.shape == (REFERENCE_N,23)
assert reference_amp.dtype == np.float32 and np.isfinite(reference_amp).all()
ref_hash = hashlib.sha256(reference_amp.tobytes(order='C')).hexdigest()
pos_hash = hashlib.sha256(positions.astype('<i8').tobytes()).hexdigest()
assert ref_hash == entry['training_reference_sha256'], (
    'STOP: AMPds2 8192-row reference differs from archived D5')
assert pos_hash == entry['training_position_sha256'], (
    'STOP: AMPds2 reference sampling positions differ from archived D5')
print('PASS: 441,483 AMPds2 training rows and exact 8,192-row D5 reference hashes')

# 4. Five controlled LIME explanations. Only perturbation/random seed varies.
runs, features, top_sets = [], [], {}
for seed in seeds:
    explainer = LimeTabularExplainer(
        training_data=reference_amp,
        feature_names=COLUMNS,
        mode='regression', categorical_features=[],
        discretize_continuous=True, discretizer='quartile',
        feature_selection='auto', sample_around_instance=False,
        kernel_width=0.75*np.sqrt(23), random_state=seed)
    explanation = explainer.explain_instance(
        data_row=x_amp_lime[0], predict_fn=score_fn_amp,
        num_features=10, num_samples=5000)
    local_r2 = float(explanation.score)
    surrogate_prediction = float(np.asarray(explanation.local_pred).reshape(-1)[0])
    target_prediction = float(np.asarray(explanation.predicted_value).reshape(-1)[0])
    assert np.isfinite([local_r2, surrogate_prediction, target_prediction]).all()
    assert abs(target_prediction - score) <= 1e-9
    coefficients = explanation.as_map()[1]
    assert len(coefficients) == 10
    weights = {int(i):float(w) for i,w in coefficients}
    assert len(weights) == 10 and all(0 <= i < 23 for i in weights)
    order = sorted(weights, key=lambda i: (-abs(weights[i]),i))
    top5 = order[:5]
    top_sets[seed] = set(top5)

    # D5 seed-42 exact numerical reproduction is a hard gate.
    if seed == 42:
        assert abs(local_r2-float(old_fidelity['lime_local_r2'])) <= 1e-6
        assert abs(surrogate_prediction-float(old_fidelity['lime_surrogate_prediction'])) <= 1e-6
        assert [FEATURES[i] for i in order] == old_coef['feature'].tolist(), (
            'STOP: Seed-42 feature identity/ranking differs from archived D5')
        expected_weights = old_coef['lime_weight'].to_numpy(dtype=np.float64)
        fresh_weights = np.asarray([weights[i] for i in order], dtype=np.float64)
        assert np.max(np.abs(fresh_weights-expected_weights)) <= 1e-6, (
            'STOP: Seed-42 regression LIME coefficients differ')
        print('PASS: Seed-42 exactly reproduces archived AMPds2 D5 regression LIME')

    runs.append(dict(
        dataset='AMPds2', window_id='AMPDS2_01', seed=seed,
        anomaly_score=score, local_r2=local_r2,
        surrogate_prediction=surrogate_prediction,
        absolute_local_prediction_error=abs(surrogate_prediction-score),
        selected_feature_count=10,
        top5_features='|'.join(FEATURES[i] for i in top5)))
    for i,name in enumerate(FEATURES):
        features.append(dict(dataset='AMPds2',window_id='AMPDS2_01',seed=seed,
            feature_index=i,feature=name,
            scaled_feature_value=float(x_amp_lime[0,i]),
            selected_in_surrogate=(i in weights),
            lime_weight=weights.get(i,None),  # unselected is NOT zero attribution
            top5=(i in top_sets[seed])))
    print(f'PASS seed={seed}: R2={local_r2:.6f}; '
          f'local_error={abs(surrogate_prediction-score):.6f}; '
          f'top5={[FEATURES[i] for i in top5]}')

assert len(runs) == 5 and len(features) == 115

# 5. Ten unique pairwise Jaccard comparisons (not independent samples).
pairs = []
for a,b in combinations(seeds,2):
    A,B = top_sets[a], top_sets[b]
    pairs.append(dict(dataset='AMPds2',window_id='AMPDS2_01',
        seed_a=a,seed_b=b,intersection_count=len(A&B),union_count=len(A|B),
        top5_jaccard=len(A&B)/len(A|B)))
assert len(pairs) == 10
j = np.asarray([v['top5_jaccard'] for v in pairs],dtype=float)
r2s = np.asarray([v['local_r2'] for v in runs],dtype=float)
summary = dict(
    dataset='AMPds2', window_id='AMPDS2_01', protocol_sha256=protocol_hash,
    method='LIME',vary='perturbation_random_state',
    training_reference='fixed_8192_training_rows',reference_seed=2027,
    reference_sha256=ref_hash, seeds=seeds, n_runs=5, n_seed_pairs=10,
    mean_pairwise_top5_jaccard=float(j.mean()),
    min_pairwise_top5_jaccard=float(j.min()),
    max_pairwise_top5_jaccard=float(j.max()),
    mean_local_r2=float(r2s.mean()),sample_sd_local_r2=float(r2s.std(ddof=1)),
    min_local_r2=float(r2s.min()),max_local_r2=float(r2s.max()),
    mean_absolute_local_error=float(np.mean([
        row['absolute_local_prediction_error'] for row in runs])))

# 6. Preserve only NEW evidence. Refuse to overwrite different existing bytes.
parent = Path(ROOT)/'SUPERVISOR_FINAL_EXPERIMENTS_2026'/'06_explanation_evaluation'
assert parent.is_dir()
output = parent/'CHECKPOINT5D_AMPDS201_LIME_PERTURBATION_5SEEDS_849e9d7e'
def csvbytes(records):
    return pd.DataFrame(records).to_csv(
        index=False,float_format='%.17g').encode('utf-8')
payloads = {
    'ampds201_lime_runs.csv': csvbytes(runs),
    'ampds201_lime_23_feature_mapping.csv': csvbytes(features),
    'ampds201_lime_pairwise_jaccard.csv': csvbytes(pairs),
    'ampds201_lime_summary.json': (
        json.dumps(summary,indent=2,sort_keys=True,allow_nan=False)+'\n').encode('utf-8')
}
for name,data in payloads.items():
    path=output/name
    if path.exists():
        assert path.read_bytes()==data, f'STOP: Existing evidence differs: {path}'
output.mkdir(exist_ok=True)
for name,data in payloads.items():
    path=output/name
    if not path.exists():
        with path.open('xb') as f: f.write(data)
    assert path.read_bytes()==data

print('\n=== CHECKPOINT 5D SUMMARY ===')
print(pd.DataFrame(runs)[[
    'seed','local_r2','absolute_local_prediction_error','top5_features'
]].to_string(index=False))
print('Mean pairwise top-5 Jaccard:', summary['mean_pairwise_top5_jaccard'])
print('Minimum Jaccard:', summary['min_pairwise_top5_jaccard'])
print('Maximum Jaccard:', summary['max_pairwise_top5_jaccard'])
print('Mean local R2:', summary['mean_local_r2'])
print('Sample SD local R2:', summary['sample_sd_local_r2'])
print('Mean absolute local prediction error:', summary['mean_absolute_local_error'])
print('Evidence directory:', output)
print('PASS: AMPDS2_01_LIME_FIVE_SEED_EXPERIMENT')
print('No detector retraining, historical artifact overwrites, SHAP, or threshold changes.')
